# Poker McPokerface — Analytics

Loads a session's JSONL logs and answers the three research questions from the brief:

1. **Best bot** — which (model, personality) combination performed best?
2. **Best LLM** — which model averaged best across all personalities?
3. **Best personality** — which personality averaged best across all models?

All three are pandas groupbys on the same flat seat-level DataFrame, so each section is a few lines of code plus a bar chart.

## 1. Imports + path setup

In [ ]:
import sys
from pathlib import Path
# Notebooks live in `notebooks/`. Make sure the project root is on
# sys.path so we can `import engine`, `import bots`, etc.
_here = Path.cwd()
_root = _here.parent if _here.name == 'notebooks' else _here
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from tracker import load_hands, load_config

## 2. Pick a session

By default we load the most recent session in `runs/`. Set `SESSION_ID` explicitly to pick a different one (e.g. the string you saw printed by `bot_arena.ipynb`).

In [ ]:
RUNS_ROOT = _root / 'runs'
SESSION_ID = None  # e.g. '20260429_180000_123456' to pin a specific session

if SESSION_ID:
    session_dir = RUNS_ROOT / SESSION_ID
else:
    sessions = sorted([p for p in RUNS_ROOT.iterdir() if p.is_dir()])
    if not sessions:
        raise SystemExit(f'No sessions found under {RUNS_ROOT}. Run bot_arena.ipynb first.')
    session_dir = sessions[-1]

print(f'Loading session: {session_dir.name}')
config = load_config(session_dir)
print(f"  started_at={config['started_at']}  hands={config['game_config'].get('num_hands')}")
for s in config['seats']:
    print(f"  {s['name']:<16s} model={s['model_id']:<14s} personality={s['personality_id']}")

## 3. Reshape into a flat seat-level DataFrame

Each `hands.jsonl` row contains a `seat_results` list. We explode that list so we have one row per (hand, seat) — the natural shape for groupbys.

In [ ]:
hands = load_hands(session_dir)
rows = []
for h in hands:
    for sr in h['seat_results']:
        rows.append({
            'hand_id': h['hand_id'],
            'name': sr['name'],
            'model_id': sr['model_id'],
            'personality_id': sr['personality_id'],
            'starting_chips': sr['starting_chips'],
            'ending_chips': sr['ending_chips'],
            'net_change': sr['net_change'],
            'folded': sr['folded'],
            # 'won' = won at least 1 chip back from the pot. Different from net_change>0 because antes count as losses.
            'won': any(w['name'] == sr['name'] for w in h['winners']),
        })
df = pd.DataFrame(rows)
print(f'{len(df)} seat-rows across {df["hand_id"].nunique()} hands and {df["name"].nunique()} bots.')
df.head()

## 4. Question 1 — Best bot (model + personality combo)

Mean chip change per hand is the headline metric, but we also show total chips won and win rate.

In [ ]:
by_bot = df.groupby(['model_id', 'personality_id', 'name']).agg(
    hands       =('hand_id',    'count'),
    mean_delta  =('net_change', 'mean'),
    total_delta =('net_change', 'sum'),
    win_rate    =('won',        'mean'),
).sort_values('mean_delta', ascending=False)
by_bot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
by_bot['mean_delta'].plot(kind='barh', ax=ax, color=['#2a9d8f' if v >= 0 else '#e76f51' for v in by_bot['mean_delta']])
ax.set_xlabel('Mean net chips per hand')
ax.set_title('Best bot — mean chip change per hand')
ax.axvline(0, color='black', linewidth=0.8)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Question 2 — Best LLM (averaged across personalities)

In [ ]:
by_llm = df.groupby('model_id').agg(
    seats      =('name',       'nunique'),
    hands      =('hand_id',    'count'),
    mean_delta =('net_change', 'mean'),
    total_delta=('net_change', 'sum'),
    win_rate   =('won',        'mean'),
).sort_values('mean_delta', ascending=False)
by_llm

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
by_llm['mean_delta'].plot(kind='barh', ax=ax,
    color=['#2a9d8f' if v >= 0 else '#e76f51' for v in by_llm['mean_delta']])
ax.set_xlabel('Mean net chips per hand')
ax.set_title('Best LLM — averaged across personalities')
ax.axvline(0, color='black', linewidth=0.8)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Question 3 — Best personality (averaged across LLMs)

In [ ]:
by_pers = df.groupby('personality_id').agg(
    seats      =('name',       'nunique'),
    hands      =('hand_id',    'count'),
    mean_delta =('net_change', 'mean'),
    total_delta=('net_change', 'sum'),
    win_rate   =('won',        'mean'),
).sort_values('mean_delta', ascending=False)
by_pers

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
by_pers['mean_delta'].plot(kind='barh', ax=ax,
    color=['#2a9d8f' if v >= 0 else '#e76f51' for v in by_pers['mean_delta']])
ax.set_xlabel('Mean net chips per hand')
ax.set_title('Best personality — averaged across LLMs')
ax.axvline(0, color='black', linewidth=0.8)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Chip trend over time

Cumulative net chip change per bot, hand by hand. Under `elimination` policy this is the chip stack itself. Under `rebuy` policy it shows true performance — rebuys reset the stack but `net_change` is computed against the start of each hand, so cumulative deltas measure underlying skill.

In [ ]:
trend = (df.sort_values('hand_id')
          .groupby('name')['net_change']
          .cumsum())
trend_df = df.assign(cum_delta=trend)

fig, ax = plt.subplots(figsize=(9, 5))
for name, sub in trend_df.groupby('name'):
    ax.plot(sub['hand_id'], sub['cum_delta'], label=name, linewidth=2, marker='', alpha=0.9)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Hand number')
ax.set_ylabel('Cumulative net chips')
ax.set_title('Chip trend per bot over the session')
ax.legend(loc='best')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()